In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from imblearn.over_sampling import RandomOverSampler
from sklearn.preprocessing import LabelEncoder
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# 读取数据
train_df = pd.read_csv('unsw-nb15/UNSW_NB15_training-set.csv')
test_df = pd.read_csv('unsw-nb15/UNSW_NB15_testing-set.csv')

# 移除无关列
train_df.drop(columns=['id', 'label'], inplace=True)
test_df.drop(columns=['id', 'label'], inplace=True)

# 检查是否有缺失值
print("Train missing values:", train_df.isnull().sum().sum())
print("Test missing values:", test_df.isnull().sum().sum())

# One-hot 编码列
cols = ['proto', 'state', 'service']

def one_hot(df, cols):
    for col in cols:
        dummies = pd.get_dummies(df[col], prefix=col, drop_first=False)
        df = pd.concat([df, dummies], axis=1)
        df.drop(col, axis=1, inplace=True)
    return df

# 合并数据进行统一处理
combined_data = pd.concat([test_df, train_df], ignore_index=True)

# 分离目标变量
target = combined_data.pop('attack_cat')

# One-hot 编码
combined_data = one_hot(combined_data, cols)

# Min-Max 归一化
def normalize(df):
    result = df.copy()
    for col in df.columns:
        max_val = df[col].max()
        min_val = df[col].min()
        if max_val > min_val:
            result[col] = (df[col] - min_val) / (max_val - min_val)
    return result

# 归一化数据
normalized_data = normalize(combined_data)

# 添加目标列
normalized_data['Class'] = target

# 检查是否有缺失值
assert not normalized_data.isnull().values.any(), "归一化后的数据存在缺失值"

# 分离特征和标签
X = normalized_data.drop(columns=['Class']).values
y = normalized_data['Class'].values
print(X.shape)
# 将目标变量转换为整数编码
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

Train missing values: 0
Test missing values: 0
(257673, 196)


In [10]:
import torch
import torch.nn as nn

# ------------------------ 改进 ASTP 模块 ------------------------
class ASTP(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        # 多尺度时间卷积 + 门控线性单元（GLU）
        self.temp_conv = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(in_channels, 16, kernel_size=32, padding='same', groups=1),  # 深度可分离卷积
                nn.GLU(dim=1)  # 门控线性单元
            ),
            nn.Sequential(
                nn.Conv1d(in_channels, 16, kernel_size=64, padding='same', dilation=2, groups=1),
                nn.GLU(dim=1)
            ),
            nn.Sequential(
                nn.Conv1d(in_channels, 16, kernel_size=96, padding='same', dilation=4, groups=1),
                nn.GLU(dim=1)
            )
        ])
        
        # SE通道注意力
        self.se_block = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(24, 12, kernel_size=1),
            nn.ReLU(),
            nn.Conv1d(12, 24, kernel_size=1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        branch_outs = [conv(x) for conv in self.temp_conv]  # 计算多尺度卷积
        merged = torch.cat(branch_outs, dim=1)  # [B, 24, L]
        return merged * self.se_block(merged)  # SE 注意力增强特征

# ------------------------ BiLSTM-AGRU（增强时序建模） ------------------------
class BiLSTM_AGRU(nn.Module):
    def __init__(self, feat_dim):
        super().__init__()
        self.bilstm = nn.LSTM(feat_dim, feat_dim // 2, num_layers=2, bidirectional=True, batch_first=True)
        self.agru = nn.GRU(feat_dim, feat_dim, num_layers=1, batch_first=True)
        self.att_weight = nn.Linear(feat_dim, 1)  # AGRU 的注意力权重
        
    def forward(self, x):
        # BiLSTM
        lstm_out, _ = self.bilstm(x)  # [B, L, D]
        
        # AGRU（使用注意力加权）
        agru_out, _ = self.agru(lstm_out)  # [B, L, D]
        att_scores = torch.sigmoid(self.att_weight(agru_out))  # [B, L, 1]
        
        return agru_out * att_scores  # 注意力加权 AGRU 输出

# ------------------------ 改进 Transformer（Local Attention + GLU） ------------------------
class LocalTransformer(nn.Module):
    def __init__(self, feat_dim, num_layers=2):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=feat_dim,
            nhead=4,
            dim_feedforward=feat_dim * 4,
            batch_first=True,
            activation=F.gelu
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # GLU 变换增强特征表达
        self.glu = nn.Sequential(
            nn.Linear(feat_dim, feat_dim * 2),
            nn.GLU(dim=-1)
        )
    
    def forward(self, x):
        x = self.transformer(x)  # 局部注意力 Transformer
        return self.glu(x)  # GLU 进行特征选择

# ------------------------ 改进 NSLKDDModel ------------------------
class NSLKDDModel(nn.Module):
    def __init__(self, input_dim=122, num_classes=5):
        super().__init__()

        # 特征提取层（ASTP）
        self.astp = ASTP(in_channels=1)
        self.pool = nn.AdaptiveMaxPool1d(input_dim // 4)  # 维度降采样
        self.bn = nn.BatchNorm1d(24)  # 特征标准化

        # 时序建模（BiLSTM-AGRU + Transformer）
        self.time_modeling = nn.Sequential(
            BiLSTM_AGRU(24),
            LocalTransformer(24, num_layers=2)
        )

        # 分类头（IEEE TDSC 2023 改进）
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(24, 128),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # 输入形状: [B, 1, 122]
        x = self.astp(x)           # [B, 24, 122]
        x = self.pool(x)           # [B, 24, 30]
        x = self.bn(x)             # 特征标准化
        x = x.permute(0, 2, 1)     # [B, 30, 24]
        x = self.time_modeling(x)  # [B, 30, 24]
        x = x.permute(0, 2, 1)     # [B, 24, 30]
        return self.classifier(x)  # [B, num_classes]


In [11]:
from sklearn import metrics
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
# 设置超参数
batch_size = 64
epochs =30   #15 0.0008 82.40  25 0.0006 82.65    25 0.0008 87.09 30 0.0008 87.21
learning_rate = 0.0008
k=10
# 交叉验证
kfold = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
oversample = RandomOverSampler(sampling_strategy='minority')

class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = nn.CrossEntropyLoss(reduction='none')(inputs, targets)
        pt = torch.exp(-ce_loss)  # 预测的概率
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

criterion = FocalLoss()
# 统计结果
oos_pred = []
# 初始化结果列表
oos_accuracies = []
last_fold_y_true = []
last_fold_y_pred = []

# 初始化模型
model = NSLKDDModel(input_dim=196, num_classes=10).to(device)
#criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 遍历折叠
# 过采样少数类
X_resampled, y_resampled = oversample.fit_resample(X, y_encoded)

#for fold, (train_idx, test_idx) in enumerate(kfold.split(X, y_encoded), start=1):
for fold, (train_idx, test_idx) in enumerate(kfold.split(X_resampled, y_resampled), start=1):
    # 划分数据集
    #X_train, X_test = X[train_idx], X[test_idx]
    #y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

    X_train, X_test = X_resampled[train_idx], X_resampled[test_idx]
    y_train, y_test = y_resampled[train_idx], y_resampled[test_idx]

    # 过采样少数类
    #X_train, y_train = oversample.fit_resample(X_train, y_train)

    # 转换为 PyTorch TensorDataset
    train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                                  torch.tensor(y_train, dtype=torch.long))
    test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                                torch.tensor(y_test, dtype=torch.long))

    # DataLoader
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    print(f"Fold {fold}: {len(train_loader.dataset)} train samples, {len(test_loader.dataset)} validation samples")

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        correct = 0
        total = 0

        # 使用 tqdm 显示进度条
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for batch_data, batch_labels in progress_bar:
            batch_data, batch_labels = batch_data.to(device), batch_labels.to(device)

            batch_data = batch_data.unsqueeze(1)  # 添加通道维度
            optimizer.zero_grad()
            outputs = model(batch_data)
            loss = criterion(outputs, batch_labels)
            loss.backward()
            optimizer.step()

            # 累积损失和准确性
            epoch_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == batch_labels).sum().item()
            total += batch_labels.size(0)

            # 更新进度条
            progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

        # 计算每轮的平均损失和准确率
        epoch_loss /= len(train_loader)
        epoch_acc = correct / total
        print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")    

    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for batch_data, batch_labels in test_loader:
            batch_data, batch_labels = batch_data.to(device), batch_labels.to(device)
            batch_data = batch_data.unsqueeze(1)  # 添加通道维度
            outputs = model(batch_data)
            _, preds = torch.max(outputs, 1)

            y_true.extend(batch_labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    # 测试集各类别数量
    # test_class_counts = Counter(y_true)
    # print("\nTest Set Class Distribution:")
    # for label, count in sorted(test_class_counts.items()):
    #     print(f"  Class {label}: {count}")

    # 计算准确率
    acc = metrics.accuracy_score(y_true, y_pred)
    oos_accuracies.append(acc)
    print(f"Fold {fold} Accuracy: {acc:.4f}")

    # 保存最后一折的结果
    if fold == kfold.get_n_splits():
        last_fold_y_true = y_true
        last_fold_y_pred = y_pred

# 保存模型
model_save_path = "modelU8.pth"
torch.save(model, model_save_path)
print(f"Complete model saved to {model_save_path}")

# 输出每一折的准确率
print("Fold Accuracies:")
for i, acc in enumerate(oos_accuracies, start=1):
    print(f"  Fold {i}: {acc:.4f}")
 

Fold 1: 315449 train samples, 35050 validation samples


Epoch 1/30: 100%|██████████| 4929/4929 [00:51<00:00, 95.97it/s, loss=0.1594] 


Epoch 1/30 - Loss: 0.3565, Accuracy: 0.7824


Epoch 2/30: 100%|██████████| 4929/4929 [00:51<00:00, 96.21it/s, loss=0.1762]


Epoch 2/30 - Loss: 0.2675, Accuracy: 0.8187


...

Epoch 28/30 - Loss: 0.1481, Accuracy: 0.8725


Epoch 29/30: 100%|██████████| 4929/4929 [00:49<00:00, 99.82it/s, loss=0.1780] 


Epoch 29/30 - Loss: 0.1476, Accuracy: 0.8729


Epoch 30/30: 100%|██████████| 4929/4929 [00:49<00:00, 99.73it/s, loss=0.0965] 


Epoch 30/30 - Loss: 0.1474, Accuracy: 0.8728
Fold 10 Accuracy: 0.8721
Complete model saved to modelU8.pth
Fold Accuracies:
  Fold 1: 0.8547
  Fold 2: 0.8670
  Fold 3: 0.8649
  Fold 4: 0.8695
  Fold 5: 0.8680
  Fold 6: 0.8723
  Fold 7: 0.8696
  Fold 8: 0.8716
  Fold 9: 0.8727
  Fold 10: 0.8721


In [12]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# 分类类别
categories = ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic','Normal', 'Reconnaissance', 'Shellcode', 'Worms']

# 混淆矩阵（最后一折的结果）
last_cm = confusion_matrix(last_fold_y_true, last_fold_y_pred, labels=range(10))

print("\nLast Fold Confusion Matrix:")
print(last_cm)

report = classification_report(last_fold_y_true, last_fold_y_pred, target_names=categories)
print("\nClassification Report:")
print(report)

# 从混淆矩阵提取所有类别的统计信息
total_samples = last_cm.sum()  # 总样本数
correct_predictions = np.trace(last_cm)  # 对角线元素的和，即正确分类的总数

# 总体准确率（直接计算）
overall_accuracy = correct_predictions / total_samples

# 初始化总体指标
overall_TP = 0
overall_FN = 0
overall_FP = 0
overall_TN = 0


# 重新计算分类报告中的 TP、FP、FN、TN
detection_rates = {}
false_positive_rates = {}

for i, category in enumerate(categories):
    TP = last_cm[i, i]
    FN = last_cm[i, :].sum() - TP
    FP = last_cm[:, i].sum() - TP
    TN = total_samples - (TP + FP + FN)

    if category != "Normal":  # 统计攻击类型的总 TP 和 FN
        overall_TP += TP
        overall_FN += FN
    else:  # 统计正常类型的 TN 和 FP
        overall_TN += TN
        overall_FP += FP

    # 每类检测率和误报率
    detection_rates[category] = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    false_positive_rates[category] = FP / (FP + TN) if (FP + TN) > 0 else 0.0

    print(f"Category: {category}")
    print(f"  Detection Rate (DR): {detection_rates[category]:.4f}")
    print(f"  False Positive Rate (FPR): {false_positive_rates[category]:.4f}")

# 总体检测率（攻击类型的 DR）和误报率（Normal 的 FPR）
overall_dr = overall_TP / (overall_TP + overall_FN) if (overall_TP + overall_FN) > 0 else 0.0
overall_fpr = overall_FP / (overall_FP + overall_TN) if (overall_FP + overall_TN) > 0 else 0.0

print("\nOverall Metrics:")
print(f"  Accuracy (Acc): {overall_accuracy:.4f}")
print(f"  Detection Rate (DR): {overall_dr:.4f}")
print(f"  False Positive Rate (FPR): {overall_fpr:.4f}")


Last Fold Confusion Matrix:
[[  19    0    0  179   39    0   31    0    0    0]
 [   0   25    1  170   30    2    0    0    4    0]
 [   4    2   90 1421   49   21   17   13   17    1]
 [   0    7   43 4031  105   46   90   71   41   18]
 [   0    0    5  220 1279    8  880   13   18    2]
 [   0    0    6   75    4 5795    3    0    2    2]
 [   0    1    2   44  374    2 8827   30   18    2]
 [   0    1    2  280    2    5    9 1092    7    1]
 [   0    1    1    7    3    4   18    8  109    0]
 [   0    0    0    0    0    0    0    0    0 9300]]

Classification Report:
                precision    recall  f1-score   support

      Analysis       0.83      0.07      0.13       268
      Backdoor       0.68      0.11      0.19       232
           DoS       0.60      0.06      0.10      1635
      Exploits       0.63      0.91      0.74      4452
       Fuzzers       0.68      0.53      0.59      2425
       Generic       0.99      0.98      0.98      5887
        Normal       0.